# Analisis de precios [SEPA](https://datos.produccion.gob.ar/dataset/sepa-precios) minoristas

**Fuentes**:

[2018-2023](https://drive.google.com/drive/folders/13GONeBs5lQCSUdBioHYk-8GhfDtIyliD)

[2024-2026](https://drive.google.com/drive/folders/1GNs9SrZ4BIoBsviBVWYYqRcsj4dwPF-I)

[Últimos meses](https://uadeeduar-my.sharepoint.com/my?id=%2Fpersonal%2Fsriverti%5Fuade%5Fedu%5Far%2FDocuments%2Fbases%5Fsepa&ct=1778176849047&or=Teams%2DHL&ga=1&LOF=1)

In [1]:
# ============================================================
# CELDA 1 — Imports y configuración (parametrizable por semestre)
# ============================================================
import os, re, gzip, zipfile, gc
import pandas as pd
import numpy as np
from pathlib import Path

# >>> ÚNICO PARÁMETRO QUE CAMBIA POR CORRIDA <
# Ejemplos válidos: "2022A", "2022B", "2023A", ..., "2026A", "2026B"
SEMESTRE = "2023A"

# Derivar año y nombre de archivo automáticamente
ANIO     = int(SEMESTRE[:4])
LETRA    = SEMESTRE[4:].upper()   # "A" o "B"
ZIP_PATH = f"/content/{SEMESTRE}.zip"

PATH_MAESTRO_PROD = "/content/Maestro de Productos Interno.xlsx"
PATH_MAESTRO_SUC  = "/content/maestro_sucursales_completo.xlsx"

WORK_DIR   = f"/content/sepa_{SEMESTRE}"
OUTPUT_DIR = f"/content/salidas_{SEMESTRE}"
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verificación
print(f"📅 Semestre a procesar: {SEMESTRE}  (año {ANIO}, semestre {LETRA})\n")
for path in [ZIP_PATH, PATH_MAESTRO_PROD, PATH_MAESTRO_SUC]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"✅ {path}  ({size_mb:.1f} MB)")
    else:
        print(f"❌ NO ENCONTRADO: {path}")

📅 Semestre a procesar: 2023A  (año 2023, semestre A)

✅ /content/2023A.zip  (1556.1 MB)
✅ /content/Maestro de Productos Interno.xlsx  (20.3 MB)
✅ /content/maestro_sucursales_completo.xlsx  (0.5 MB)


In [2]:
# ============================================================
# CELDA 2 — Definición de la canasta
# ============================================================
CANASTA = {
    '7790742363008': ('Leche entera 1L',         20, 'Lácteos'),
    '7791337007628': ('Yogur 190g',               8, 'Lácteos'),
    '7791337061361': ('Queso Casancrem 290g',     2, 'Lácteos'),
    '7793940052002': ('Manteca 100g',             2, 'Lácteos'),
    '7791337007253': ('Cindor 1L',                4, 'Lácteos'),
    '7790272001029': ('Aceite girasol 1,5L',      2, 'Almacén'),
    '7790070433114': ('Arroz 500g',               2, 'Almacén'),
    '7790070320285': ('Fideos 500g',              4, 'Almacén'),
    '7792180140708': ('Harina leudante 1kg',      2, 'Almacén'),
    '7792710000182': ('Yerba 500g',               2, 'Almacén'),
    '7790550000157': ('Café 250g',                1, 'Almacén'),
    '7790040143234': ('Chocolinas 250g',          4, 'Almacén'),
    '7790072002080': ('Sal fina 500g',            1, 'Almacén'),
    '7790895000232': ('Coca Cola lata',           8, 'Bebidas'),
    '7790895067570': ('Coca Sin Azúcar 2,25L',    4, 'Bebidas'),
    '7798062548716': ('Agua Levite 500ml',        8, 'Bebidas'),
    '7793147118860': ('Cerveza lata',             6, 'Bebidas'),
    '7798074864675': ('Vino Malbec 750ml',        2, 'Bebidas'),
    '7790132098459': ('Lavandina 1L',             2, 'Limpieza'),
    '7791290794054': ('Detergente 300ml',         2, 'Limpieza'),
    '7793253003500': ('Limpiador Poett 900ml',    2, 'Limpieza'),
    '7791293047447': ('Shampoo 400ml',            1, 'Higiene'),
    '7791293045948': ('Acondicionador 340ml',     1, 'Higiene'),
    '7791293051208': ('Jabón tocador 90g',        4, 'Higiene'),
    '7791293049557': ('Antitranspirante',         2, 'Higiene'),
    '7891024183083': ('Hilo dental',              1, 'Higiene'),
    '7790770601899': ('Toallas femeninas x16',    2, 'Higiene'),
    '7790250015840': ('Papel higiénico',          2, 'Higiene'),
    '7790580327415': ('Rocklets 40g',             2, 'Snacks'),
    '7790580716707': ('Saladix 100g',             2, 'Snacks'),
}

CANASTA_EANS_RAW    = set(CANASTA.keys())
CANASTA_EANS_LSTRIP = {e.lstrip('0') for e in CANASTA_EANS_RAW}
print(f"Canasta: {len(CANASTA)} productos")

Canasta: 30 productos


In [3]:
# ============================================================
# CELDA 3 — Descomprimir el zip e inventariar
# ============================================================
# Limpiar working dir por si quedó algo de una corrida anterior
import shutil
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

print(f"Descomprimiendo {os.path.basename(ZIP_PATH)}...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(WORK_DIR)

# Buscar todos los .csv.gz recursivamente (incluye variantes como "...COMPLETOb.csv.gz")
archivos = []
for root, _, files in os.walk(WORK_DIR):
    for f in files:
        if f.endswith('.csv.gz'):
            full = os.path.join(root, f)
            archivos.append((f, full, os.path.getsize(full)/1024/1024))
archivos.sort()

print(f"\nArchivos .csv.gz encontrados: {len(archivos)}")
for nombre, _, mb in archivos:
    print(f"  {nombre:<50}  {mb:>8.1f} MB")

if len(archivos) == 0:
    print("\n⚠️ No se encontraron archivos. Revisar contenido del zip.")

Descomprimiendo 2023A.zip...

Archivos .csv.gz encontrados: 12
  012023_pais_parte1COMPLETO.csv.gz                      136.2 MB
  012023_pais_parte2COMPLETO.csv.gz                      135.5 MB
  022023_pais_parte1COMPLETO.csv.gz                      133.1 MB
  022023_pais_parte2COMPLETO.csv.gz                      120.9 MB
  032023_pais_parte1COMPLETO.csv.gz                      136.5 MB
  032023_pais_parte2COMPLETO.csv.gz                      128.5 MB
  042023_pais_parte1COMPLETO.csv.gz                      146.3 MB
  042023_pais_parte2COMPLETO.csv.gz                      130.3 MB
  052023_pais_parte1COMPLETO.csv.gz                      136.5 MB
  052023_pais_parte2COMPLETO.csv.gz                      136.7 MB
  062023_pais_parte1COMPLETO.csv.gz                      120.2 MB
  062023_pais_parte2COMPLETO.csv.gz                      136.5 MB


In [4]:
# ============================================================
# CELDA 4 — Cobertura temporal (leyendo solo headers)
# ============================================================
patron_fecha = re.compile(r'^precio_(\d{8})$')
todas_fechas_set = set()
cobertura = []

for nombre, full, mb in archivos:
    df_h = pd.read_csv(full, compression='gzip', sep=',', nrows=0)
    fechas_arch = sorted(
        pd.to_datetime(patron_fecha.match(c).group(1), format='%Y%m%d')
        for c in df_h.columns if patron_fecha.match(c)
    )
    todas_fechas_set.update(fechas_arch)
    cobertura.append({
        'archivo': nombre,
        'tamaño_MB': round(mb, 1),
        'fecha_min': fechas_arch[0].date() if fechas_arch else None,
        'fecha_max': fechas_arch[-1].date() if fechas_arch else None,
        'dias_cubiertos': len(fechas_arch),
    })

cobertura_df = pd.DataFrame(cobertura)
todas_fechas = sorted(todas_fechas_set)

print("=== COBERTURA POR ARCHIVO ===")
print(cobertura_df.to_string(index=False))

if todas_fechas:
    print(f"\n=== COBERTURA GLOBAL ===")
    print(f"Desde:           {todas_fechas[0].date()}")
    print(f"Hasta:           {todas_fechas[-1].date()}")
    print(f"Días con datos:  {len(todas_fechas)}")
    print(f"Días esperados:  {(todas_fechas[-1] - todas_fechas[0]).days + 1}")

    dias_faltantes = sorted(set(pd.date_range(todas_fechas[0], todas_fechas[-1])) - set(todas_fechas))
    print(f"Días faltantes:  {len(dias_faltantes)}")
    if dias_faltantes:
        for d in dias_faltantes[:30]:
            print(f"  - {d.date()}")
else:
    print("⚠️ No se detectaron fechas")

=== COBERTURA POR ARCHIVO ===
                          archivo  tamaño_MB  fecha_min  fecha_max  dias_cubiertos
012023_pais_parte1COMPLETO.csv.gz      136.2 2023-01-01 2023-01-15              15
012023_pais_parte2COMPLETO.csv.gz      135.5 2023-01-16 2023-01-31              16
022023_pais_parte1COMPLETO.csv.gz      133.1 2023-02-01 2023-02-15              15
022023_pais_parte2COMPLETO.csv.gz      120.9 2023-02-16 2023-02-28              13
032023_pais_parte1COMPLETO.csv.gz      136.5 2023-03-01 2023-03-15              15
032023_pais_parte2COMPLETO.csv.gz      128.5 2023-03-16 2023-03-31              16
042023_pais_parte1COMPLETO.csv.gz      146.3 2023-04-01 2023-04-15              15
042023_pais_parte2COMPLETO.csv.gz      130.3 2023-04-16 2023-04-30              15
052023_pais_parte1COMPLETO.csv.gz      136.5 2023-05-01 2023-05-15              15
052023_pais_parte2COMPLETO.csv.gz      136.7 2023-05-16 2023-05-31              16
062023_pais_parte1COMPLETO.csv.gz      120.2 2023-06-01 2

In [5]:
# ============================================================
# CELDA 5 — Cargar maestros y enriquecer la canasta
# ============================================================
maestro_prod = pd.read_excel(PATH_MAESTRO_PROD)
maestro_suc  = pd.read_excel(PATH_MAESTRO_SUC)

print(f"Maestro productos:  {len(maestro_prod):,} filas")
print(f"Maestro sucursales: {len(maestro_suc):,} filas\n")

maestro_prod['ean_clean'] = maestro_prod['producto_sepa_id'].astype(str).str.lstrip('0')
canasta_match = maestro_prod[maestro_prod['ean_clean'].isin(CANASTA_EANS_LSTRIP)].copy()

print(f"=== MATCH DE CANASTA EN MAESTRO ===")
print(f"Productos de la canasta encontrados: {len(canasta_match)} / {len(CANASTA)}\n")

match_eans = set(canasta_match['ean_clean'])
no_encontrados = [e for e in CANASTA_EANS_RAW if e.lstrip('0') not in match_eans]
if no_encontrados:
    print(f"⚠️ EANs no encontrados en maestro:")
    for e in no_encontrados:
        print(f"  - {e}: {CANASTA[e][0]}")

cols_show = ['producto_sepa_id','producto_descripcion','producto_marca',
             'rubro','categoria','subcategoria','proveedor','producto_blacklist']
print("\n=== INFO ENRIQUECIDA DE LA CANASTA ===")
print(canasta_match[cols_show].to_string(index=False))

print(f"\n=== REGIONES ===")
print(maestro_suc['REGION'].value_counts())

Maestro productos:  176,702 filas
Maestro sucursales: 3,611 filas

=== MATCH DE CANASTA EN MAESTRO ===
Productos de la canasta encontrados: 30 / 30


=== INFO ENRIQUECIDA DE LA CANASTA ===
producto_sepa_id                                        producto_descripcion producto_marca      rubro               categoria          subcategoria                                                                                     proveedor  producto_blacklist
   7790070320285               Fideos Spaghetti Fortificados Favorita 500 Gr       Favorita    Almacén            Pastas Secas          Pastas Secas                                                                  MOLINOS RIO DE LA PLATA S.A.                 0.0
   7790072002080                          Sal Fina en Estuche Celusal 500 Gr        Celusal    Almacén                Aderezos        Sal y Pimienta                                                                IND.QUIM. Y MINERAS TIMBO S.A.                 0.0
   7790132098459      

In [ ]:
# ============================================================
# CELDA 6 — Procesar archivos: filtrar canasta y convertir wide→long
# ============================================================
def normalizar_ean(s):
    if pd.isna(s):
        return None
    s = str(s).strip().lstrip('0')
    return s if s else '0'

acumulador = []

# Primera pasada: leer todo SIN dividir, después decidimos el factor
print("Primera pasada (sin dividir): acumulando datos crudos...")

for i, (nombre, full, mb) in enumerate(archivos, 1):
    print(f"[{i}/{len(archivos)}] {nombre} ({mb:.0f} MB)")

    try:
        df = pd.read_csv(full, compression='gzip', sep=',', dtype=str)
    except Exception as e:
        print(f"   ⚠️ No se pudo leer: {e}")
        continue

    cols_precio = [c for c in df.columns if patron_fecha.match(c)]
    if len(cols_precio) == 0:
        del df; gc.collect(); continue

    cols_id = ['id_comercio','id_bandera','id_sucursal','sucursales_provincia','id_producto']
    if not all(c in df.columns for c in cols_id):
        del df; gc.collect(); continue

    df['ean_norm'] = df['id_producto'].apply(normalizar_ean)
    df_canasta = df[df['ean_norm'].isin(CANASTA_EANS_LSTRIP)].copy()
    print(f"   filas totales: {len(df):>10,} | filas canasta: {len(df_canasta):>8,}")

    if len(df_canasta) == 0:
        del df; gc.collect(); continue

    df_long = df_canasta.melt(
        id_vars=cols_id + ['ean_norm'],
        value_vars=cols_precio,
        var_name='fecha_col',
        value_name='precio_raw'
    )

    df_long['fecha'] = pd.to_datetime(
        df_long['fecha_col'].str.replace('precio_', '', regex=False),
        format='%Y%m%d'
    )
    df_long.drop(columns=['fecha_col'], inplace=True)

    # Por ahora dejamos el precio crudo, sin dividir
    df_long['precio_raw_num'] = pd.to_numeric(df_long['precio_raw'], errors='coerce')
    df_long.drop(columns=['precio_raw'], inplace=True)
    df_long = df_long[df_long['precio_raw_num'].notna() & (df_long['precio_raw_num'] > 5)]

    acumulador.append(df_long)
    del df, df_canasta
    gc.collect()

if len(acumulador) == 0:
    raise RuntimeError("No se obtuvo ninguna fila de la canasta. Revisar archivos.")

canasta_long = pd.concat(acumulador, ignore_index=True)
del acumulador; gc.collect()

# Eliminar duplicados
antes = len(canasta_long)
canasta_long = canasta_long.drop_duplicates(
    subset=['fecha','id_comercio','id_bandera','id_sucursal','ean_norm'],
    keep='first'
)
duplicados = antes - len(canasta_long)
if duplicados > 0:
    print(f"\n⚠️ Se eliminaron {duplicados:,} filas duplicadas")

# --- Detección de factor de precio usando la canasta como ground truth ---
# El producto más barato y universal de la canasta es la sal Celusal 500g.
# Verificamos su precio mediano para decidir el factor.
print("\n🔎 Detectando factor de precio usando productos conocidos de la canasta...")

# EANs de productos baratos y estables (ajuste razonable: sal, fideos, lavandina)
EANS_REFERENCIA = ['7790072002080',  # Sal Celusal 500g
                   '7790070320285',  # Fideos Favorita 500g
                   '7790132098459']  # Lavandina Ayudín 1L
eans_ref_norm = {e.lstrip('0') for e in EANS_REFERENCIA}

referencia = canasta_long[canasta_long['ean_norm'].isin(eans_ref_norm)]
mediana_ref = referencia['precio_raw_num'].median() if len(referencia) else 0

# Criterio: estos productos típicamente cuestan entre $30 y $5000 en cualquier año
# Si la mediana ya está en ese rango → factor 1
# Si está 100x más alta → factor 100
# Si está 10000x más alta → factor 10000 (poco probable pero por las dudas)
if 30 <= mediana_ref <= 5000:
    FACTOR_PRECIO = 1
elif 3000 <= mediana_ref <= 500000:
    FACTOR_PRECIO = 100
elif mediana_ref > 500000:
    FACTOR_PRECIO = 10000
else:
    # Mediana sospechosamente baja, asumir factor 1 y warnear
    FACTOR_PRECIO = 1
    print(f"   ⚠️ Mediana de referencia inusualmente baja ({mediana_ref:.2f})")

print(f"   Mediana cruda de productos de referencia (sal/fideos/lavandina): {mediana_ref:,.2f}")
print(f"   → FACTOR aplicado: dividir por {FACTOR_PRECIO}")

# Aplicar el factor
canasta_long['precio'] = canasta_long['precio_raw_num'] / FACTOR_PRECIO
canasta_long.drop(columns=['precio_raw_num'], inplace=True)

print(f"\n✅ Total observaciones diarias de la canasta: {len(canasta_long):,}")
print(f"Rango: {canasta_long['fecha'].min().date()} → {canasta_long['fecha'].max().date()}")

# Sanity check: precio mediano por producto de referencia
print(f"\nSanity check de precios (mediana global por producto):")
sanity = (canasta_long[canasta_long['ean_norm'].isin(eans_ref_norm)]
          .groupby('ean_norm')['precio'].median().round(2))
for e, p in sanity.items():
    nombre_prod = next((CANASTA[ean][0] for ean in CANASTA if ean.lstrip('0') == e), e)
    print(f"  {nombre_prod}: ${p:,.2f}")

Primera pasada (sin dividir): acumulando datos crudos...
[1/12] 012023_pais_parte1COMPLETO.csv.gz (136 MB)
   filas totales: 12,464,587 | filas canasta:   22,246
[2/12] 012023_pais_parte2COMPLETO.csv.gz (136 MB)
   filas totales: 12,585,856 | filas canasta:   22,648
[3/12] 022023_pais_parte1COMPLETO.csv.gz (133 MB)
   filas totales: 12,742,383 | filas canasta:   24,998
[4/12] 022023_pais_parte2COMPLETO.csv.gz (121 MB)
   filas totales: 13,519,594 | filas canasta:   25,036
[5/12] 032023_pais_parte1COMPLETO.csv.gz (136 MB)
   filas totales: 13,636,563 | filas canasta:   25,477
[6/12] 032023_pais_parte2COMPLETO.csv.gz (128 MB)
   filas totales: 13,674,649 | filas canasta:   28,483
[7/12] 042023_pais_parte1COMPLETO.csv.gz (146 MB)
   filas totales: 13,505,626 | filas canasta:   28,803
[8/12] 042023_pais_parte2COMPLETO.csv.gz (130 MB)
   filas totales: 13,435,066 | filas canasta:   30,434
[9/12] 052023_pais_parte1COMPLETO.csv.gz (137 MB)
   filas totales: 13,167,581 | filas canasta:   31,00

In [ ]:
# ============================================================
# CELDA 7 — Enriquecer con maestros (productos y sucursales)
# ============================================================
maestro_prod_slim = (canasta_match[['ean_clean','producto_descripcion','producto_marca',
                                     'rubro','categoria','subcategoria','proveedor']]
                     .rename(columns={'ean_clean':'ean_norm',
                                      'producto_descripcion':'descripcion',
                                      'producto_marca':'marca'}))
canasta_long = canasta_long.merge(maestro_prod_slim, on='ean_norm', how='left')

maestro_suc_slim = maestro_suc[['id_comercio','id_bandera','id_sucursal',
                                 'sucursales_nombre','sucursales_tipo',
                                 'sucursales_localidad','PROVINCIA','REGION']].copy()
for c in ['id_comercio','id_bandera','id_sucursal']:
    maestro_suc_slim[c] = maestro_suc_slim[c].astype(str)
    canasta_long[c] = canasta_long[c].astype(str)

canasta_long = canasta_long.merge(maestro_suc_slim,
                                   on=['id_comercio','id_bandera','id_sucursal'],
                                   how='left')

canasta_long.rename(columns={'PROVINCIA':'provincia','REGION':'region'}, inplace=True)

columnas_finales = ['fecha','ean_norm','descripcion','marca',
                    'rubro','categoria','subcategoria','proveedor',
                    'id_comercio','id_bandera','id_sucursal',
                    'sucursales_nombre','sucursales_tipo','sucursales_localidad',
                    'provincia','region','sucursales_provincia','precio']
columnas_finales = [c for c in columnas_finales if c in canasta_long.columns]
canasta_long = canasta_long[columnas_finales]

print(f"✅ Canasta enriquecida: {len(canasta_long):,} filas")
print(f"Provincias detectadas: {canasta_long['provincia'].nunique()}")
print(f"Regiones detectadas:   {canasta_long['region'].nunique()}")
print(f"Sucursales únicas:     {canasta_long.groupby(['id_comercio','id_bandera','id_sucursal']).ngroups:,}")
print(f"\nMuestra:")
print(canasta_long.head(5).to_string())

In [ ]:
# ============================================================
# CELDA 8 — Resúmenes (sin descargar todavía)
# ============================================================
# A) Serie temporal diaria nacional
serie_diaria = (canasta_long.groupby(['fecha','ean_norm','descripcion'])['precio']
                .agg(['mean','median','count','min','max'])
                .round(2)
                .reset_index())

# B) Precio mensual por (producto, provincia)
canasta_long['mes'] = canasta_long['fecha'].dt.to_period('M').astype(str)
precio_mes_prov = (canasta_long.groupby(['mes','ean_norm','descripcion','provincia'])['precio']
                   .mean().round(2).reset_index())

# C) Precio mensual por (producto, region)
precio_mes_reg = (canasta_long.groupby(['mes','ean_norm','descripcion','region'])['precio']
                  .mean().round(2).reset_index())

# D) Canasta mensual valorada por provincia
qty_map = {ean: v[1] for ean, v in CANASTA.items()}
qty_map_lstrip = {k.lstrip('0'): v for k, v in qty_map.items()}
precio_mes_prov['cantidad'] = precio_mes_prov['ean_norm'].map(qty_map_lstrip)
precio_mes_prov['subtotal'] = precio_mes_prov['precio'] * precio_mes_prov['cantidad']
canasta_mensual_prov = (precio_mes_prov.groupby(['mes','provincia'])
                        .agg(canasta_total=('subtotal','sum'),
                             productos_disponibles=('ean_norm','nunique'))
                        .reset_index()
                        .sort_values(['mes','canasta_total']))

# E) Canasta mensual valorada por región
precio_mes_reg['cantidad'] = precio_mes_reg['ean_norm'].map(qty_map_lstrip)
precio_mes_reg['subtotal'] = precio_mes_reg['precio'] * precio_mes_reg['cantidad']
canasta_mensual_reg = (precio_mes_reg.groupby(['mes','region'])
                       .agg(canasta_total=('subtotal','sum'),
                            productos_disponibles=('ean_norm','nunique'))
                       .reset_index()
                       .sort_values(['mes','canasta_total']))

print(f"✅ Resúmenes calculados:")
print(f"   Serie diaria: {len(serie_diaria):,} filas")
print(f"   Canasta por provincia: {len(canasta_mensual_prov):,} filas")
print(f"   Canasta por región:    {len(canasta_mensual_reg):,} filas")
print(f"\nMeses cubiertos en la canasta: {sorted(canasta_long['mes'].unique())}")

In [ ]:
# ============================================================
# CELDA 9 — Canasta nacional ponderada por población (Censo 2022)
# ============================================================
POBLACION_2022 = {
    'Buenos Aires':         17_523_996,
    'Córdoba':               3_840_905,
    'Santa Fe':              3_544_908,
    'CABA':                  3_121_707,
    'Mendoza':               2_043_540,
    'Tucumán':               1_731_820,
    'Salta':                 1_441_351,
    'Entre Ríos':            1_425_578,
    'Misiones':              1_278_873,
    'Corrientes':            1_212_696,
    'Chaco':                 1_129_606,
    'Santiago del Estero':   1_060_906,
    'San Juan':                822_853,
    'Jujuy':                   811_611,
    'Río Negro':               750_768,
    'Neuquén':                 710_814,
    'Formosa':                 607_419,
    'Chubut':                  592_621,
    'San Luis':                542_069,
    'Catamarca':               429_562,
    'La Rioja':                383_865,
    'La Pampa':                361_859,
    'Santa Cruz':              337_226,
    'Tierra del Fuego':        185_732,
}

POBLACION_TOTAL = sum(POBLACION_2022.values())
print(f"Población total considerada: {POBLACION_TOTAL:,}")
print(f"Provincias con peso: {len(POBLACION_2022)}\n")

# ---- Normalización de nombres de provincias ----
# Maneja variantes históricas y typos que aparecen en distintos años del maestro
NORMALIZAR_PROVINCIA = {
    'Provincia de Buenos Aires':       'Buenos Aires',
    'Ciudad Autónoma de Buenos Aires': 'CABA',
    'Ciudad de Buenos Aires':          'CABA',
    'San juan':                        'San Juan',
    'Entre Rios':                      'Entre Ríos',
    'Cordoba':                         'Córdoba',
    'Rio Negro':                       'Río Negro',
    'Neuquen':                         'Neuquén',
    'Tucuman':                         'Tucumán',
}

canasta_mensual_prov['provincia'] = (
    canasta_mensual_prov['provincia'].replace(NORMALIZAR_PROVINCIA)
)
# Re-agregar por si quedaron duplicados con nombres distintos que ahora son el mismo
canasta_mensual_prov = (canasta_mensual_prov
                       .groupby(['mes','provincia'], as_index=False)
                       .agg(canasta_total=('canasta_total','mean'),
                            productos_disponibles=('productos_disponibles','max')))

# También normalizar la canasta_long (que se exporta al parquet) para coherencia
canasta_long['provincia'] = canasta_long['provincia'].replace(NORMALIZAR_PROVINCIA)

print("✅ Nombres de provincias normalizados\n")

# Verificar matching de nombres
provincias_dataset = set(canasta_mensual_prov['provincia'].dropna().unique())
provincias_pob = set(POBLACION_2022.keys())

en_dataset_no_pob = provincias_dataset - provincias_pob
en_pob_no_dataset = provincias_pob - provincias_dataset

if en_dataset_no_pob:
    print(f"⚠️ En el dataset pero sin peso poblacional: {en_dataset_no_pob}")
if en_pob_no_dataset:
    print(f"⚠️ Con peso poblacional pero sin datos: {en_pob_no_dataset}")
if not en_dataset_no_pob and not en_pob_no_dataset:
    print("✅ Todas las provincias matchean entre dataset y población\n")

# Pesos
pob_df = pd.DataFrame([
    {'provincia': p, 'poblacion': pob, 'peso': pob / POBLACION_TOTAL}
    for p, pob in POBLACION_2022.items()
])

# Canasta nacional ponderada por mes
canasta_nac_pond = canasta_mensual_prov.merge(pob_df, on='provincia', how='inner')
canasta_nac_pond['canasta_x_peso'] = (
    canasta_nac_pond['canasta_total'] * canasta_nac_pond['peso']
)

canasta_nacional = (canasta_nac_pond.groupby('mes')
                    .agg(canasta_nacional_ponderada=('canasta_x_peso','sum'),
                         provincias_con_dato=('provincia','nunique'),
                         poblacion_cubierta=('poblacion','sum'))
                    .reset_index())
canasta_nacional['cobertura_poblacional_%'] = (
    canasta_nacional['poblacion_cubierta'] / POBLACION_TOTAL * 100
).round(2)
canasta_nacional['canasta_nacional_ponderada'] = canasta_nacional['canasta_nacional_ponderada'].round(2)

# Promedio simple para comparar
canasta_promedio_simple = (canasta_mensual_prov
                           .groupby('mes')['canasta_total']
                           .mean().round(2).reset_index()
                           .rename(columns={'canasta_total':'canasta_promedio_simple'}))
canasta_nacional = canasta_nacional.merge(canasta_promedio_simple, on='mes')
canasta_nacional['diferencia_%'] = (
    (canasta_nacional['canasta_nacional_ponderada'] /
     canasta_nacional['canasta_promedio_simple'] - 1) * 100
).round(2)

# Variación mensual
canasta_nacional = canasta_nacional.sort_values('mes').reset_index(drop=True)
canasta_nacional['variacion_mensual_%'] = (
    canasta_nacional['canasta_nacional_ponderada'].pct_change() * 100
).round(2)

canasta_nacional['semestre'] = SEMESTRE

print("=== CANASTA NACIONAL PONDERADA POR POBLACIÓN ===")
print(canasta_nacional[['mes','canasta_nacional_ponderada','variacion_mensual_%',
                        'canasta_promedio_simple','diferencia_%',
                        'cobertura_poblacional_%']].to_string(index=False))

In [ ]:
# ============================================================
# CELDA 10 — Exportar TODO a Excel + Parquet y descargar
# ============================================================
output_xlsx    = os.path.join(OUTPUT_DIR, f"canasta_{SEMESTRE}_serie.xlsx")
output_parquet = os.path.join(OUTPUT_DIR, f"canasta_{SEMESTRE}_long.parquet")

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as w:
    cobertura_df.to_excel(w, sheet_name='cobertura_temporal', index=False)
    canasta_match[['producto_sepa_id','producto_descripcion','producto_marca',
                   'rubro','categoria','subcategoria','proveedor']
                 ].to_excel(w, sheet_name='canasta_definicion', index=False)
    serie_diaria.to_excel(w, sheet_name='serie_diaria_nacional', index=False)
    canasta_mensual_prov.to_excel(w, sheet_name='canasta_mes_provincia', index=False)
    canasta_mensual_reg.to_excel(w, sheet_name='canasta_mes_region', index=False)
    canasta_nacional.to_excel(w, sheet_name='canasta_nacional_ponderada', index=False)
    pob_df.sort_values('peso', ascending=False).to_excel(
        w, sheet_name='pesos_poblacionales', index=False)

canasta_long.to_parquet(output_parquet, index=False)

print(f"✅ Excel resumen:    {output_xlsx}")
print(f"✅ Parquet detalle:  {output_parquet}")
print(f"   ({os.path.getsize(output_parquet)/1024/1024:.1f} MB)")

from google.colab import files
files.download(output_xlsx)
files.download(output_parquet)

In [ ]:
# ============================================================
# CELDA DIAGNÓSTICA — verificar si cambió el factor a mitad del semestre
# ============================================================
import pandas as pd

# Mediana del precio de la sal Celusal mes a mes
EAN_SAL = '7790072002080'.lstrip('0')
sal = canasta_long[canasta_long['ean_norm'] == EAN_SAL].copy()
sal['mes'] = sal['fecha'].dt.to_period('M').astype(str)

print("=== Precio mediano de SAL Celusal 500g mes a mes ===")
print(sal.groupby('mes')['precio'].agg(['median','min','max','count']).round(2))

print("\n=== Precio mediano de LAVANDINA Ayudín 1L mes a mes ===")
EAN_LAV = '7790132098459'.lstrip('0')
lav = canasta_long[canasta_long['ean_norm'] == EAN_LAV].copy()
lav['mes'] = lav['fecha'].dt.to_period('M').astype(str)
print(lav.groupby('mes')['precio'].agg(['median','min','max','count']).round(2))

print("\n=== Precio mediano de FIDEOS Favorita 500g mes a mes ===")
EAN_FID = '7790070320285'.lstrip('0')
fid = canasta_long[canasta_long['ean_norm'] == EAN_FID].copy()
fid['mes'] = fid['fecha'].dt.to_period('M').astype(str)
print(fid.groupby('mes')['precio'].agg(['median','min','max','count']).round(2))